# Track 3 — Industrial (Infineon): EDA for GPT-2 Fine-Tuning

**Goal:** Understand the structure of MOSFET / IGBT / IC process sequences so we can design an optimal fine-tuning strategy for GPT-2.

**Key questions answered here:**
1. How large is the vocabulary and how does it differ between families?
2. How long are sequences, and how variable are they?
3. What are the transition patterns (bigram structure)?
4. Where do steps appear in the sequence (positional bias)?
5. What is the best tokenization and training format for GPT-2?

---

In [ ]:
import csv
import math
import random
from pathlib import Path
from collections import Counter, defaultdict

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

DATA_DIR = Path('training_data')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

FAMILY_COLORS = {'MOSFET': '#4C72B0', 'IGBT': '#DD8452', 'IC': '#55A868'}

# ── helper ──────────────────────────────────────────────────────────────────
def load_variants(family: str) -> dict:
    """Load variant CSV; return {seq_id: [step, ...]}"""
    fname = DATA_DIR / f'{family}_variants.csv'
    seqs: dict = {}
    with open(fname) as f:
        for row in csv.DictReader(f):
            sid = row['SEQUENCE_ID']
            seqs.setdefault(sid, []).append(row['STEP'])
    return seqs

families = {name: load_variants(name) for name in ['MOSFET', 'IGBT', 'IC']}
print('Loaded families:', {k: len(v) for k, v in families.items()})

## 1. Sequence Length Distribution

Sequence length determines GPT-2's context window requirements and training batch size.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)

for ax, (name, seqs) in zip(axes, families.items()):
    lengths = [len(v) for v in seqs.values()]
    ax.hist(lengths, bins=20, color=FAMILY_COLORS[name], edgecolor='white', linewidth=0.5)
    ax.set_title(f'{name}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Sequence length (steps)')
    ax.set_ylabel('Count')
    stats = f'min={min(lengths)}  max={max(lengths)}\nmean={np.mean(lengths):.1f}  std={np.std(lengths):.1f}'
    ax.text(0.97, 0.97, stats, transform=ax.transAxes,
            ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

plt.suptitle('Sequence Length Distribution per Family', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_length_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ IGBT sequences are ~23 steps longer than IC — GPT-2 context window must cover ~160 tokens')

## 2. Vocabulary Analysis — Shared vs. Family-Specific Steps

Understanding which steps are family-exclusive is critical for deciding whether to train one shared model or three separate models.

In [ ]:
vocab = {}
for name, seqs in families.items():
    vocab[name] = set(s for seq in seqs.values() for s in seq)

mosfet_v, igbt_v, ic_v = vocab['MOSFET'], vocab['IGBT'], vocab['IC']

shared_all   = mosfet_v & igbt_v & ic_v
mosfet_igbt  = (mosfet_v & igbt_v) - ic_v
mosfet_ic    = (mosfet_v & ic_v) - igbt_v
igbt_ic      = (igbt_v & ic_v) - mosfet_v
mosfet_only  = mosfet_v - igbt_v - ic_v
igbt_only    = igbt_v - mosfet_v - ic_v
ic_only      = ic_v - mosfet_v - igbt_v

categories = {
    'Shared (all 3)': (len(shared_all), '#888888'),
    'MOSFET+IGBT': (len(mosfet_igbt), '#9370DB'),
    'MOSFET+IC': (len(mosfet_ic), '#20B2AA'),
    'IGBT+IC': (len(igbt_ic), '#F4A460'),
    'MOSFET only': (len(mosfet_only), FAMILY_COLORS['MOSFET']),
    'IGBT only': (len(igbt_only), FAMILY_COLORS['IGBT']),
    'IC only': (len(ic_only), FAMILY_COLORS['IC']),
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
labels = list(categories.keys())
sizes  = [v[0] for v in categories.values()]
colors = [v[1] for v in categories.values()]
bars = ax1.bar(labels, sizes, color=colors, edgecolor='white', linewidth=0.8)
for bar, sz in zip(bars, sizes):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, str(sz),
             ha='center', va='bottom', fontsize=10, fontweight='bold')
ax1.set_title('Step Vocabulary Overlap', fontsize=13, fontweight='bold')
ax1.set_ylabel('Number of unique steps')
ax1.tick_params(axis='x', rotation=30)
ax1.set_ylim(0, max(sizes) * 1.15)

# Pie chart of total vocab split
pie_labels = ['Shared all 3', 'MOSFET only', 'IGBT only', 'IC only', 'Pairwise shared']
pairwise = len(mosfet_igbt) + len(mosfet_ic) + len(igbt_ic)
pie_sizes = [len(shared_all), len(mosfet_only), len(igbt_only), len(ic_only), pairwise]
pie_colors = ['#888888', FAMILY_COLORS['MOSFET'], FAMILY_COLORS['IGBT'], FAMILY_COLORS['IC'], '#C0A0D0']
ax2.pie(pie_sizes, labels=pie_labels, colors=pie_colors, autopct='%1.0f%%',
        startangle=90, textprops={'fontsize': 10})
ax2.set_title(f'Total vocab = {sum(pie_sizes)} steps', fontsize=13, fontweight='bold')

plt.suptitle('Vocabulary Distribution Across Families', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_vocab_overlap.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Total unique steps (full vocab): {len(mosfet_v | igbt_v | ic_v)}')
print(f'Shared by all 3 families: {len(shared_all)} ({len(shared_all)/(len(mosfet_v|igbt_v|ic_v))*100:.0f}% of vocab)')
print('\n→ 47% of vocabulary is shared — a single shared model with family conditioning is justified.')

### Family-exclusive steps (important for model diagnostics)

In [ ]:
exclusive = {
    'MOSFET only': sorted(mosfet_only),
    'IGBT only': sorted(igbt_only),
    'IC only': sorted(ic_only),
}
for fam, steps in exclusive.items():
    print(f'\n=== {fam} ({len(steps)} steps) ===')
    for s in steps:
        print(f'  {s}')

## 3. Step Frequency Distribution

High-frequency steps dominate the corpus — the model must learn both common backbone steps and rare family-specific steps.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 18))

for ax, (name, seqs) in zip(axes, families.items()):
    all_steps = [s for seq in seqs.values() for s in seq]
    cnt = Counter(all_steps)
    # Top 40 steps by frequency
    top = cnt.most_common(40)
    labels, counts = zip(*top)
    counts_per_seq = [c / len(seqs) for c in counts]
    
    bars = ax.barh(range(len(labels)), counts_per_seq, color=FAMILY_COLORS[name], alpha=0.85)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('Average occurrences per sequence')
    ax.set_title(f'{name} — Top 40 steps by frequency', fontsize=12, fontweight='bold')
    ax.axvline(x=1.0, color='red', linestyle='--', linewidth=0.8, alpha=0.6, label='once per seq')
    ax.legend(fontsize=8)

plt.suptitle('Step Frequency (avg per sequence)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_step_freq.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Transition Matrix (Bigram Heatmap)

The bigram structure reveals the near-deterministic process grammar — critical for understanding what GPT-2 must learn.

In [ ]:
def build_transition_matrix(seqs, top_n=40):
    """Build normalized transition matrix for the top_n most frequent steps."""
    all_steps = [s for seq in seqs.values() for s in seq]
    top_steps = [s for s, _ in Counter(all_steps).most_common(top_n)]
    idx = {s: i for i, s in enumerate(top_steps)}
    
    mat = np.zeros((top_n, top_n))
    for seq in seqs.values():
        for a, b in zip(seq, seq[1:]):
            if a in idx and b in idx:
                mat[idx[a], idx[b]] += 1
    
    # Row-normalize
    row_sums = mat.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    mat_norm = mat / row_sums
    return mat_norm, top_steps

fig, axes = plt.subplots(1, 3, figsize=(22, 8))

for ax, (name, seqs) in zip(axes, families.items()):
    mat, steps = build_transition_matrix(seqs, top_n=30)
    im = ax.imshow(mat, cmap='Blues', aspect='auto', vmin=0, vmax=1)
    ax.set_xticks(range(len(steps)))
    ax.set_yticks(range(len(steps)))
    ax.set_xticklabels(steps, rotation=90, fontsize=6)
    ax.set_yticklabels(steps, fontsize=6)
    ax.set_title(f'{name} Transition Matrix (top 30 steps)', fontsize=10, fontweight='bold')
    plt.colorbar(im, ax=ax, shrink=0.6)

plt.suptitle('Step Transition Probabilities (row = current step, col = next step)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_transition_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ High sparsity with strong diagonal bands = process grammar is nearly deterministic.')
print('→ GPT-2 will learn this structure quickly; diversity comes from optional-step variation axes.')

## 5. Bigram Entropy — How Deterministic Is Each Step?

Low entropy = next step is predictable. High entropy = model needs to choose from multiple valid continuations.

In [ ]:
def bigram_entropy(seqs):
    """Returns dict: step -> Shannon entropy of next-step distribution."""
    trans = defaultdict(Counter)
    for seq in seqs.values():
        for a, b in zip(seq, seq[1:]):
            trans[a][b] += 1
    entropies = {}
    for step, nxt in trans.items():
        total = sum(nxt.values())
        probs = [c / total for c in nxt.values()]
        entropies[step] = -sum(p * math.log2(p) for p in probs if p > 0)
    return entropies

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, seqs) in zip(axes, families.items()):
    ent = bigram_entropy(seqs)
    # Top-20 highest-entropy steps
    top = sorted(ent.items(), key=lambda x: -x[1])[:20]
    labels, vals = zip(*top)
    ax.barh(range(len(labels)), vals, color=FAMILY_COLORS[name], alpha=0.85)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('Shannon Entropy (bits)')
    ax.set_title(f'{name}\nTop-20 highest-entropy steps', fontsize=11, fontweight='bold')
    ax.axvline(x=1.0, color='red', linestyle='--', linewidth=0.8, alpha=0.5)

plt.suptitle('Bigram Entropy per Step\n(measures ambiguity in next-step prediction)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_bigram_entropy.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary stats
for name, seqs in families.items():
    ent = bigram_entropy(seqs)
    vals = list(ent.values())
    high = [(s, f'{e:.2f}') for s, e in sorted(ent.items(), key=lambda x: -x[1])[:5]]
    print(f'{name}: mean_entropy={np.mean(vals):.3f} bits, max={max(vals):.3f} bits')
    print(f'  Top-5 ambiguous: {high}')

print('\n→ Most steps have entropy < 1 bit — the grammar is very constrained.')
print('→ High-entropy steps (optional measurements, RCA alternates) are exactly the variation axes in the rules.')

## 6. Positional Analysis — Where Does Each Step Appear?

Normalized position (0=start, 1=end) reveals the block structure and whether positional encoding matters.

In [ ]:
# Key steps that distinguish the families and represent each block
KEY_STEPS = {
    'MOSFET': [
        'RECEIVE WAFER LOT', 'SUBSTRATE CHECK', 'EPITAXIAL DEPOSITION',
        'THERMAL OXIDATION', 'IMPLANT WELL', 'DEPOSIT POLYSILICON',
        'IMPLANT LDD', 'DEPOSIT INTERLAYER DIELECTRIC',
        'DEPOSIT BARRIER METAL', 'DEPOSIT METAL 1',
        'DEPOSIT PASSIVATION', 'CURE PASSIVATION',
        'WAFER SORT TEST', 'SHIP LOT'
    ],
    'IGBT': [
        'RECEIVE WAFER LOT', 'EPITAXIAL WAFER CHECK',
        'THERMAL OXIDATION', 'IMPLANT P BODY', 'IMPLANT N BUFFER',
        'DEPOSIT FIELD OXIDE', 'IMPLANT CHANNEL STOP',
        'DEPOSIT INTERLAYER DIELECTRIC', 'DEPOSIT BARRIER METAL',
        'DEPOSIT METAL 1', 'DEPOSIT PASSIVATION', 'CURE PASSIVATION',
        'BREAKDOWN VOLTAGE TEST', 'WAFER SORT TEST', 'SHIP LOT'
    ],
    'IC': [
        'RECEIVE WAFER LOT', 'GRINDING WAFER BACKSIDE',
        'THERMAL OXIDATION', 'DEPOSIT PAD OXIDE',
        'DEPOSIT POLYSILICON', 'IMPLANT N-TYPE',
        'DEPOSIT INTERLAYER DIELECTRIC', 'DEPOSIT TUNGSTEN SEED',
        'DEPOSIT METAL 1', 'DEPOSIT PASSIVATION', 'CURE PASSIVATION',
        'DEPOSIT BACKSIDE PROTECTION', 'WAFER SORT TEST', 'SHIP LOT'
    ]
}

fig, axes = plt.subplots(1, 3, figsize=(18, 7))

for ax, (name, seqs) in zip(axes, families.items()):
    step_pos = defaultdict(list)
    for seq in seqs.values():
        n = len(seq)
        for i, s in enumerate(seq):
            step_pos[s].append(i / (n - 1))
    
    key = KEY_STEPS[name]
    # Filter to steps actually in data
    key = [s for s in key if s in step_pos]
    
    means  = [np.mean(step_pos[s]) for s in key]
    stds   = [np.std(step_pos[s]) for s in key]
    
    y = range(len(key))
    ax.barh(y, means, xerr=stds, color=FAMILY_COLORS[name], alpha=0.8,
            error_kw=dict(ecolor='black', capsize=3, linewidth=1))
    ax.set_yticks(y)
    ax.set_yticklabels(key, fontsize=8)
    ax.set_xlim(0, 1)
    ax.set_xlabel('Normalized position in sequence (0=start, 1=end)')
    ax.set_title(f'{name}\nKey step positions (mean ± std)', fontsize=11, fontweight='bold')
    ax.axvline(x=0.5, color='gray', linestyle='--', linewidth=0.7, alpha=0.5)

plt.suptitle('Step Positional Analysis — Block Structure Visible', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_positional.png', dpi=150, bbox_inches='tight')
plt.show()
print('→ Very low std = steps appear at fixed positions → GPT-2 positional embeddings will strongly help.')

## 7. IC vs IGBT vs MOSFET — Structural Comparison

This is the core EDA for deciding how to train the model.

In [ ]:
# Process block coverage — which blocks/steps mark the boundary of each phase
BLOCK_MARKERS = {
    'Prefix / Lot Start': ['RECEIVE WAFER LOT', 'LOT IDENTIFICATION'],
    'Pre-Process Clean': ['PRE CLEAN WAFER', 'WAFER CLEAN PRE PROCESS', 'HF DIP'],
    'Family Prep (MOSFET)': ['SUBSTRATE CHECK', 'EPITAXIAL DEPOSITION', 'EPITAXY ANNEAL'],
    'Family Prep (IGBT)': ['EPITAXIAL WAFER CHECK', 'EPITAXIAL LAYER PREP'],
    'Family Prep (IC)': ['GRINDING WAFER BACKSIDE', 'ETCH WET BACKSIDE'],
    'First Oxidation': ['THERMAL OXIDATION'],
    'Litho Cycles': ['SPIN COAT PHOTORESIST', 'DEVELOP PHOTORESIST'],
    'ILD Block': ['DEPOSIT INTERLAYER DIELECTRIC', 'CMP DIELECTRIC'],
    'Via Block': ['DEPOSIT BARRIER METAL', 'FILL VIA METAL', 'FILL VIA TUNGSTEN'],
    'Metal Block': ['DEPOSIT METAL 1', 'DEPOSIT TOP METAL'],
    'Passivation': ['DEPOSIT PASSIVATION', 'CURE PASSIVATION'],
    'Backside': ['DEPOSIT BACKSIDE METAL', 'BACKSIDE ANNEAL'],
    'Test Suite': ['PARAMETRIC TEST', 'WAFER SORT TEST', 'YIELD ANALYSIS'],
    'Ship': ['SHIP LOT'],
}

print('=== Block coverage per family ===')
print(f'{"Block":<35} {"MOSFET":^10} {"IGBT":^10} {"IC":^10}')
print('-' * 70)
for block, markers in BLOCK_MARKERS.items():
    row = []
    for name in ['MOSFET', 'IGBT', 'IC']:
        present = any(m in vocab[name] for m in markers)
        row.append('✓' if present else '✗')
    print(f'{block:<35} {row[0]:^10} {row[1]:^10} {row[2]:^10}')

In [ ]:
# Lithography cycle counts
print('\n=== Litho levels per family ===')
for name, seqs in families.items():
    max_levels = []
    for seq in seqs.values():
        levels = set()
        for s in seq:
            if s.startswith('ALIGN MASK LEVEL '):
                try:
                    levels.add(int(s.split()[-1]))
                except ValueError:
                    pass
        max_levels.append(max(levels) if levels else 0)
    cnt = Counter(max_levels)
    print(f'{name}: litho_levels_dist = {dict(sorted(cnt.items()))}')

In [ ]:
# Implant step analysis — the IGBT has the most diverse implant profile
implant_steps = [s for s in (mosfet_v | igbt_v | ic_v) if 'IMPLANT' in s]
implant_steps.sort()

print('\n=== Implant steps per family ===')
print(f'{"Step":<40} {"MOSFET":^10} {"IGBT":^10} {"IC":^10}')
print('-' * 70)
for step in implant_steps:
    row = ['✓' if step in vocab[n] else ' ' for n in ['MOSFET', 'IGBT', 'IC']]
    print(f'{step:<40} {row[0]:^10} {row[1]:^10} {row[2]:^10}')

## 8. Optional Step Variation Analysis

The generation rules define 11 variation axes. Here we measure how often each optional step actually appears.

In [ ]:
OPTIONAL_STEPS = {
    'POST EXPOSE BAKE': 'Optional in litho cycle (IC-style)',
    'HARD BAKE': 'Optional in litho cycle',
    'DRY WAFER': 'Optional after HF DIP',
    'DRY WAFER BACKSIDE': 'Optional after HF DIP',
    'EPITAXIAL REWORK CHECK': 'IGBT only, optional',
    'PRE ANNEAL CHECK': 'Optional before RTA',
    'MEASURE SURFACE PARTICLES': 'Optional measurement',
    'MEASURE SURFACE DEFECTS': 'Optional measurement',
    'BACKSIDE CLEAN': 'Optional in pre-process clean',
    'FRONTSIDE CLEAN': 'Optional in pre-process clean',
    'GATE OXIDE PREP': 'MOSFET optional',
    'GATE OXIDE GROWTH': 'MOSFET optional',
    'ANNEAL OXIDE': 'IC optional',
    'CMP METAL': 'Via CMP optional',
    'CMP VIA FILL': 'Via CMP optional',
    'BACKSIDE THINNING CHECK': 'IC only, optional',
    'DEPOSIT BACKSIDE PROTECTION': 'IC only, optional',
    'FINAL OXIDE CHECK': 'Optional final inspection',
    'FINAL ELECTRICAL TEST PREP': 'Optional final inspection',
    'PACKAGE PREPARATION': 'IC only, optional',
}

fig, axes = plt.subplots(1, 3, figsize=(18, 8))

for ax, (name, seqs) in zip(axes, families.items()):
    present_steps = [(step, desc) for step, desc in OPTIONAL_STEPS.items() if step in vocab[name]]
    rates = []
    labels = []
    for step, desc in present_steps:
        rate = sum(1 for seq in seqs.values() if step in seq) / len(seqs)
        rates.append(rate)
        labels.append(step)
    
    # Sort by rate
    order = np.argsort(rates)[::-1]
    rates = [rates[i] for i in order]
    labels = [labels[i] for i in order]
    
    bars = ax.barh(range(len(labels)), rates, color=FAMILY_COLORS[name], alpha=0.8)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlim(0, 1.05)
    ax.set_xlabel('Fraction of sequences containing step')
    ax.set_title(f'{name}\nOptional step appearance rate', fontsize=11, fontweight='bold')
    ax.axvline(x=0.5, color='gray', linestyle='--', linewidth=0.7, alpha=0.5)
    
    for bar, rate in zip(bars, rates):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{rate:.0%}', va='center', fontsize=7)

plt.suptitle('Optional Step Presence Rates\n(measures variety in training data)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_optional_steps.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Forbidden-Rule Signature Analysis

The 10 forbidden rules define ordering constraints. Here we visualize **which step pairs** each rule monitors — this directly informs what context window length GPT-2 needs.

In [ ]:
# The forbidden rules and their look-back window sizes (from generation_rules.md)
FORBIDDEN_RULES = {
    'RULE_DEP_NO_CLEAN': {
        'window': 12,
        'desc': 'Any deposition must be preceded by a clean step within 12 steps',
        'examples': [('DEPOSIT BARRIER METAL', 'must follow'), ('CLEAN AFTER VIA ETCH', 'within 12 steps')]
    },
    'RULE_METAL_ETCH_NO_LITHO': {
        'window': 15,
        'desc': 'Metal etch must be preceded by complete litho (EXPOSE+DEVELOP) within 15 steps',
        'examples': [('METAL ETCH', 'requires'), ('EXPOSE LITHO + DEVELOP', 'within 15 steps')]
    },
    'RULE_ETCH_NO_MASK': {
        'window': 12,
        'desc': 'Any etch must be preceded by DEVELOP PHOTORESIST within 12 steps',
        'examples': []
    },
    'RULE_LITHO_LEVEL_SKIP': {
        'window': 'full',
        'desc': 'Litho levels must be sequential (no level skip)',
        'examples': []
    },
    'RULE_IMPLANT_NO_MASK': {
        'window': 15,
        'desc': 'Implant must be preceded by oxide etch or develop within 15 steps',
        'examples': []
    },
    'RULE_CMP_NO_DEP': {
        'window': 6,
        'desc': 'CMP must be preceded by a deposition step within 6 steps',
        'examples': []
    },
    'RULE_PAD_OPEN_BEFORE_DEP': {
        'window': 'full',
        'desc': 'PAD WINDOW LITHO must appear after DEPOSIT PASSIVATION + CURE PASSIVATION',
        'examples': []
    },
    'RULE_TEST_BEFORE_PASSIVATION': {
        'window': 'full',
        'desc': 'Electrical tests must appear after CURE PASSIVATION',
        'examples': []
    },
    'RULE_SHIP_BEFORE_TEST': {
        'window': 'full',
        'desc': 'SHIP LOT must appear after WAFER SORT TEST',
        'examples': []
    },
    'RULE_BACKSIDE_BEFORE_PASSIVATION': {
        'window': 'full',
        'desc': 'DEPOSIT BACKSIDE METAL must appear after CURE PASSIVATION',
        'examples': []
    },
}

fig, ax = plt.subplots(figsize=(12, 5))

rules = list(FORBIDDEN_RULES.keys())
windows = [r['window'] if r['window'] != 'full' else 160 for r in FORBIDDEN_RULES.values()]
colors = ['#E74C3C' if w == 160 else '#3498DB' for w in windows]
labels = [str(w) if w != 160 else 'full seq' for w in windows]

bars = ax.barh(range(len(rules)), windows, color=colors, alpha=0.8)
ax.set_yticks(range(len(rules)))
ax.set_yticklabels(rules, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Look-back window (number of steps)')
ax.set_title('Forbidden Rule Look-back Windows\n(determines required GPT-2 context)', fontsize=12, fontweight='bold')

for bar, label in zip(bars, labels):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            label, va='center', fontsize=9, fontweight='bold')

red_patch = mpatches.Patch(color='#E74C3C', label='Requires full sequence context')
blue_patch = mpatches.Patch(color='#3498DB', label='Local window (6-15 steps)')
ax.legend(handles=[red_patch, blue_patch], fontsize=9)

plt.tight_layout()
plt.savefig('eda_rule_windows.png', dpi=150, bbox_inches='tight')
plt.show()

print('→ 6 of 10 rules need global (full-sequence) context.')
print('→ GPT-2 context window must cover the FULL sequence (~160 tokens max).')
print('→ GPT-2 default context (1024 tokens) is MORE than enough for 198-step vocabularies.')

## 10. Corpus Statistics Summary for GPT-2 Fine-Tuning

In [ ]:
total_seqs = sum(len(s) for s in families.values())
total_tokens = sum(len(s) for seqs in families.values() for s in seqs.values())
full_vocab = mosfet_v | igbt_v | ic_v

print('=' * 60)
print('  CORPUS STATISTICS SUMMARY')
print('=' * 60)
print(f'  Total sequences:          {total_seqs:,}')
print(f'  Total steps (tokens):     {total_tokens:,}')
print(f'  Full step vocabulary:     {len(full_vocab)} unique steps')
print(f'  Shared across all 3:      {len(shared_all)} steps ({len(shared_all)/len(full_vocab)*100:.0f}%)')
print(f'  Family-exclusive:         {len(mosfet_only)+len(igbt_only)+len(ic_only)} steps')
print()
for name, seqs in families.items():
    lengths = [len(v) for v in seqs.values()]
    tokens = sum(lengths)
    print(f'  {name:8}: {len(seqs):,} seqs | '
          f'len {min(lengths)}-{max(lengths)} (avg {np.mean(lengths):.0f}) | '
          f'{len(vocab[name])} vocab steps')
print('=' * 60)

## 11. GPT-2 Fine-Tuning Strategy

Based on the EDA above, here are the concrete recommendations:

In [ ]:
strategy = """
╔══════════════════════════════════════════════════════════════════╗
║          GPT-2 FINE-TUNING STRATEGY (based on EDA)             ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  1. TOKENIZATION                                                 ║
║     • Each process step = ONE custom token (not subwords)        ║
║     • Add 3 family-conditioning special tokens:                  ║
║       [MOSFET], [IGBT], [IC]                                     ║
║     • Add [SEP] between steps (or use newline separator)         ║
║     • Add [EOS] at SHIP LOT (GPT-2 stop condition)               ║
║     • Vocabulary size: 198 steps + ~10 special tokens = ~210     ║
║                                                                  ║
║  2. SEQUENCE FORMAT for training                                 ║
║     [MOSFET] RECEIVE WAFER LOT | LOT IDENTIFICATION | ...       ║
║     → prepend family token as BOS                                ║
║     → pipe-separate steps within one line                        ║
║     → max seq len ≈ 160 tokens (fits GPT-2 context easily)       ║
║                                                                  ║
║  3. MODEL CHOICE                                                 ║
║     • GPT-2 small (124M params) → sufficient for 198-token vocab  ║
║     • Train ONE shared model (47% shared vocab justifies this)    ║
║     • Alternative: 3 family-specific models (simpler, faster)     ║
║                                                                  ║
║  4. TRAINING DATA                                                ║
║     • 3,000 sequences total (1,000 per family)                   ║
║     • Total tokens: ~388,000 (enough for fine-tuning)            ║
║     • Generate more with generate_sequences.py if needed         ║
║     • Recommended: 5,000-10,000 per family for robust anomaly    ║
║       detection (scaling experiment mentioned in rules)           ║
║                                                                  ║
║  5. TRAINING OBJECTIVE                                           ║
║     • Causal LM (next-step prediction) — directly matches Task 1 ║
║     • Loss = cross-entropy over step tokens only                  ║
║     • For anomaly detection (Task 3): use per-sequence perplexity ║
║       as the anomaly score (low PPL = valid, high PPL = anomaly)  ║
║                                                                  ║
║  6. KEY EDA INSIGHTS for training                                ║
║     • Max bigram entropy < 2 bits → grammar is learnable fast     ║
║     • Steps have fixed positional bias → positional embeddings    ║
║       in GPT-2 will pick this up automatically                    ║
║     • IGBT has 6 litho levels; MOSFET/IC have 4 → family token   ║
║       is critical to disambiguate which litho level comes next    ║
║     • 6 of 10 forbidden rules require global context →            ║
║       full sequence (not sliding window) training is essential    ║
║                                                                  ║
║  7. EVALUATION HOOKS                                             ║
║     • Task 1 (next-step): sample top-5 logits at cut point       ║
║     • Task 2 (completion): greedy/beam decode from cut point      ║
║     • Task 3 (anomaly): compute mean token log-prob of full seq   ║
║       → threshold at 2-3 std below mean of valid sequences        ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
"""
print(strategy)

## 12. Training Data Preparation Example

Code to format sequences as GPT-2 training text.

In [ ]:
FAMILY_TOKENS = {'MOSFET': '[MOSFET]', 'IGBT': '[IGBT]', 'IC': '[IC]'}
SEP = ' | '
EOS = '[EOS]'

def format_sequence_for_gpt2(family: str, steps: list) -> str:
    """Format a process sequence as GPT-2 training text."""
    return FAMILY_TOKENS[family] + ' ' + SEP.join(steps) + ' ' + EOS

# Show example sequences
for name, seqs in families.items():
    example_seq = list(seqs.values())[0]
    formatted = format_sequence_for_gpt2(name, example_seq)
    print(f'\n{name} example (first 200 chars):')
    print(formatted[:200] + '...')
    print(f'  Total text length: {len(formatted)} chars, {len(example_seq)+2} tokens')

In [ ]:
# Build the full custom vocabulary for the tokenizer
all_steps = sorted(mosfet_v | igbt_v | ic_v)
special_tokens = ['[MOSFET]', '[IGBT]', '[IC]', '[EOS]', '[PAD]', '[SEP]']

# Show vocabulary size
print(f'Custom step tokens: {len(all_steps)}')
print(f'Special tokens: {special_tokens}')
print(f'Total custom vocab to add to GPT-2 tokenizer: {len(all_steps) + len(special_tokens)}')
print()
print('First 20 step tokens (alphabetical):')
for i, s in enumerate(all_steps[:20]):
    print(f'  {i+1:3}. {s}')

In [ ]:
# Export vocabulary to a text file for use with HuggingFace tokenizer
vocab_output = Path('gpt2_finetuning_vocab.txt')
with open(vocab_output, 'w') as f:
    for tok in special_tokens:
        f.write(tok + '\n')
    for step in all_steps:
        f.write(step + '\n')
print(f'Saved {len(special_tokens) + len(all_steps)} tokens to {vocab_output}')

# Export formatted training corpus
corpus_output = Path('gpt2_training_corpus.txt')
total_lines = 0
with open(corpus_output, 'w') as f:
    for name, seqs in families.items():
        for steps in seqs.values():
            line = format_sequence_for_gpt2(name, steps)
            f.write(line + '\n')
            total_lines += 1
print(f'Saved {total_lines} training sequences to {corpus_output}')

## 13. IC vs IGBT Structural Difference — Key Insight for Model Design

This is the most important comparison for understanding why the model needs family conditioning.

In [ ]:
# Show aligned block structure side by side
BLOCK_STEPS = [
    ('LOT START', ['RECEIVE WAFER LOT', 'LOT IDENTIFICATION', 'INITIAL WAFER INSPECTION']),
    ('FAMILY PREP', ['SUBSTRATE CHECK', 'EPITAXIAL DEPOSITION', 'EPITAXIAL WAFER CHECK',
                     'GRINDING WAFER BACKSIDE', 'ETCH WET BACKSIDE']),
    ('FIRST OXIDATION', ['THERMAL OXIDATION', 'DEPOSIT PAD OXIDE', 'ANNEAL OXIDE']),
    ('IMPLANT 1', ['IMPLANT WELL', 'IMPLANT P BODY']),
    ('IMPLANT 2', ['IMPLANT LDD', 'IMPLANT N BUFFER', 'IMPLANT N-TYPE']),
    ('IMPLANT 3', ['IMPLANT SOURCE DRAIN', 'IMPLANT CHANNEL STOP']),
    ('IMPLANT 4', ['IMPLANT DRAIN / CATHODE REGION', 'IMPLANT SOURCE REGION']),
    ('VIA FILL', ['FILL VIA METAL', 'FILL VIA TUNGSTEN', 'DEPOSIT TUNGSTEN SEED']),
    ('PASSIVATION', ['DEPOSIT PASSIVATION', 'CURE PASSIVATION']),
    ('BACKSIDE', ['DEPOSIT BACKSIDE METAL', 'DEPOSIT BACKSIDE PROTECTION']),
    ('TEST', ['THRESHOLD VOLTAGE TEST', 'BREAKDOWN VOLTAGE TEST', 'WAFER SORT TEST']),
    ('SUFFIX', ['PACKAGE PREPARATION', 'SHIP LOT']),
]

print(f'{"Block":<20} {"MOSFET steps":^30} {"IGBT steps":^30} {"IC steps":^30}')
print('=' * 115)
for block_name, steps in BLOCK_STEPS:
    by_fam = {n: [] for n in ['MOSFET', 'IGBT', 'IC']}
    for step in steps:
        for n in ['MOSFET', 'IGBT', 'IC']:
            if step in vocab[n]:
                by_fam[n].append(step.replace('IMPLANT ', 'IMP '))
    mosfet_str = ', '.join(by_fam['MOSFET']) or '—'
    igbt_str   = ', '.join(by_fam['IGBT'])   or '—'
    ic_str     = ', '.join(by_fam['IC'])     or '—'
    print(f'{block_name:<20} {mosfet_str:<30} {igbt_str:<30} {ic_str:<30}')

---

## Summary & Key Takeaways

| Question | Finding | Implication for GPT-2 |
|---|---|---|
| Vocabulary size? | 198 unique steps total, 94 shared | Extend GPT-2 tokenizer with 204 custom tokens |
| How long are sequences? | MOSFET ~125, IGBT ~148, IC ~115 | GPT-2 1024-token context is more than sufficient |
| Is grammar deterministic? | Mean bigram entropy < 1 bit | Model will converge quickly, even on 1K seqs |
| Which rules need global context? | 6 of 10 forbidden rules | Train on full sequences, not sliding windows |
| Does family matter? | IGBT has 6 litho levels vs 4; different implants | Always prepend `[FAMILY]` token as BOS |
| Best anomaly score? | Per-sequence mean log-probability | Low log-prob = unusual sequence = anomaly |
| IC vs IGBT key difference? | IC: early backside grind + tungsten fill; IGBT: 4 distinct implants + field oxide | Family-specific prep block is the hardest part to get right |

**The most critical design choice: use a single shared model with `[FAMILY]` conditioning token — not three separate models — because 47% of the vocabulary is shared and the backbone structure is identical across all three families.**